Import Libraries

In [3]:
import pandas as pd 
import sqlite3 
import json

Create the Customer Data

In [4]:
customers = pd.DataFrame({ 
    "CustomerID": [ 
        "C001", "C002", "C003", "C004", 
        "C004", "C005", "C006", "C007", 
        "C008", "C009", "C010" 
    ], 
    "CustomerName": [ 
        "Amina", "Brian", "Carol", "David", 
        "David", "Esther", "Faith", "George", 
        "Halima", "Ian", "Jane" 
    ], 
    "Segment": [ 
        "Retail", "Corporate", "Retail", "SME", 
        "SME", "Corporate", "SME", "Retail", 
        "Corporate", "Retail", "SME" 
    ], 
    "RegionCode": [ 
        "NRB", "MSA", "NRB", "KSM", 
        "KSM", "NKR", "NRB", "MSA", 
        "KSM", "NKR", "NRB" 
    ] 
}) 
 
customers 

customers.to_csv( 
    "customers.csv", 
    index=False 
) 

Create the Orders Data 

In [5]:
orders = pd.DataFrame({ 
    "OrderID": [ 
        "O001", "O002", "O003", "O004", "O005", 
        "O006", "O007", "O008", "O009", "O010", 
        "O011", "O012", "O013", "O014", "O015" 
    ], 
    "CustomerID": [ 
        "C001", "C002", "C001", "C003", "C004", 
        "C005", "C006", "C002", "C007", "C008", 
        "C009", "C010", "C006", "C003", "C999" 
    ], 
    "OrderAmount": [ 
        4500, 12500, 3200, 6800, 9100, 
        15000, 7600, 8300, 2500, 11000, 
        5400, 6200, 4300, 7200, 9900 
    ], 
    "ProductCategory": [ 
        "Electronics", "Office", "Home", "Electronics", "Office", 
        "Electronics", "Home", "Office", "Home", "Electronics", 
        "Office", "Home", "Electronics", "Office", "Electronics" 
    ] 
}) 
 
orders 

orders.to_csv( 
    "orders.csv", 
    index=False 
) 

Create the JSON Region Lookup

In [6]:
regions = [ 
    {"RegionCode": "NRB", "RegionName": "Nairobi"}, 
    {"RegionCode": "MSA", "RegionName": "Mombasa"}, 
    {"RegionCode": "KSM", "RegionName": "Kisumu"}, 
    {"RegionCode": "NKR", "RegionName": "Nakuru"} 
] 
 
with open("regions.json", "w") as file: 
    json.dump(regions, file, indent=4) 
    

Inspecting the Sources 

In [7]:
customers = pd.read_csv("customers.csv") 
orders = pd.read_csv("orders.csv") 

customers.head() 
orders.head()

customers.shape
orders.shape

(15, 4)

Q1. What does one row represent in customers.csv? 
-- One row represents all the available information about a single customer (Their ID, name, Segmentand Region)

Q2. What does one row represent in orders.csv? 
-- One row represents the details about an order made by a customer 

Q3. Are the two datasets at the same aggregation grain? Explain briefly. 
-- No they are not at the same aggregation grain since one row in each dataset represents different things. 

Detecting Key problems 

In [8]:
# Check for duplicate customer IDs 
customers["CustomerID"].duplicated().sum()

customers[
    customers["CustomerID"].duplicated(keep=False)
]

,CustomerID,CustomerName,Segment,RegionCode
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Q4. Why can a duplicate CustomerID be dangerous when joining customer and order tables? 
-- The same customer would be treated as two different people (If a customer were to make an order, a standard join would create a duplicate of that order in the final dataset, therefore inflating the revenue gained for the company) 

In [9]:
# Check Unmatched Customer IDs
set(orders["CustomerID"]) - set(customers["CustomerID"])


{'C999'}

Q5. Which CustomerID appears in orders but not in customers?
-- C999

Q6. Why should an unmatched record be investigated rather than silently deleted? 
-- Because like in our case a valid transaction would be lost and would misrepresent other metrics like total revenue as a result 

Create the SQLite Database 

In [10]:
conn = sqlite3.connect("retail_lab.db") 

customers.to_sql( 
    "customers", 
    conn, 
    if_exists="replace", 
    index=False 
) 
 
orders.to_sql( 
    "orders", 
    conn, 
    if_exists="replace", 
    index=False 
) 

15

Test the Database

In [11]:
query = """ 
SELECT * 
FROM customers 
LIMIT 5; 
""" 
 
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C002,Brian,Corporate,MSA
2,C003,Carol,Retail,NRB
3,C004,David,SME,KSM
4,C004,David,SME,KSM


Basic SELECT

In [12]:
query = """ 
SELECT * 
FROM orders;
""" 

pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O002,C002,12500,Office
2,O003,C001,3200,Home
3,O004,C003,6800,Electronics
4,O005,C004,9100,Office
5,O006,C005,15000,Electronics
6,O007,C006,7600,Home
7,O008,C002,8300,Office
8,O009,C007,2500,Home
9,O010,C008,11000,Electronics


Write a query that returns only OrderID, CustomerID and OrderAmount

In [13]:
query = """ 
    SELECT OrderID, CustomerID, OrderAmount 
    FROM orders
""" 
 
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount
0,O001,C001,4500
1,O002,C002,12500
2,O003,C001,3200
3,O004,C003,6800
4,O005,C004,9100
5,O006,C005,15000
6,O007,C006,7600
7,O008,C002,8300
8,O009,C007,2500
9,O010,C008,11000


Write a query that returns only customers in the Retail segment. 

In [14]:
query = """ 
    SELECT * 
    FROM customers 
    WHERE Segment = 'Retail'
""" 
 
pd.read_sql_query(query, conn)

,CustomerID,CustomerName,Segment,RegionCode
0,C001,Amina,Retail,NRB
1,C003,Carol,Retail,NRB
2,C007,George,Retail,MSA
3,C009,Ian,Retail,NKR


Write a query that returns orders with OrderAmount greater than KSh 8,000 

In [15]:
query = """ 
    SELECT * 
    FROM orders 
    WHERE OrderAmount > 8000
""" 
 
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O002,C002,12500,Office
1,O005,C004,9100,Office
2,O006,C005,15000,Electronics
3,O008,C002,8300,Office
4,O010,C008,11000,Electronics
5,O015,C999,9900,Electronics


Write a query that returns only Electronics orders. 

In [16]:
query = """ 
    SELECT * 
    FROM orders 
    WHERE ProductCategory = 'Electronics'
""" 
 
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O001,C001,4500,Electronics
1,O004,C003,6800,Electronics
2,O006,C005,15000,Electronics
3,O010,C008,11000,Electronics
4,O013,C006,4300,Electronics
5,O015,C999,9900,Electronics


Write a query that returns Electronics orders worth more than KSh 8,000.

In [17]:
query = """ 
    SELECT * 
    FROM orders 
    WHERE ProductCategory = 'Electronics' AND OrderAmount > 8000
""" 
 
pd.read_sql_query(query, conn)

,OrderID,CustomerID,OrderAmount,ProductCategory
0,O006,C005,15000,Electronics
1,O010,C008,11000,Electronics
2,O015,C999,9900,Electronics


GROUP BY and Aggregate Functions 

Total Sales 

In [18]:
query = """
    SELECT 
    SUM(OrderAmount) AS TotalSales 
FROM orders; 
"""
pd.read_sql_query(query, conn)


,TotalSales
0,113500


Sales by Product Category 

In [21]:
query = """
    SELECT 
    ProductCategory, 
    SUM(OrderAmount) AS TotalSales 
FROM orders 
GROUP BY ProductCategory; 
"""

pd.read_sql_query(query, conn)

,ProductCategory,TotalSales
0,Electronics,51500
1,Home,19500
2,Office,42500


3 KPIs for each product category 

In [22]:
query = """ 
    SELECT 
        ProductCategory AS "Product Category", 
        COUNT(OrderID) AS "Number of Orders", 
        SUM(OrderAmount) AS "Total Sales", 
        AVG(OrderAmount) AS "Average Order Value" 
    FROM orders 
    GROUP BY ProductCategory
""" 
 
pd.read_sql_query(query, conn)

,Product Category,Number of Orders,Total Sales,Average Order Value
0,Electronics,6,51500,8583.333333
1,Home,4,19500,4875.000000
2,Office,5,42500,8500.000000


Q7. Write two short observations about category performance. Refer to actual values from your output. 

-- The least performing category was 'Home' with a value of ksh 19,500/= Total sales, with only 4 total orders 

-- The best performing category was 'Electronics' with a value of ksh 51,500/= total sales, and the most orders at 6 

 JOIN Customers and Orders

In [26]:
query = """ 
SELECT 
o.OrderID, 
o.CustomerID, 
c.CustomerName, 
c.Segment, 
o.OrderAmount 
FROM orders o 
LEFT JOIN customers c 
ON o.CustomerID = c.CustomerID; 
""" 
joined = pd.read_sql_query(query, conn) 
joined

,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O005,C004,David,SME,9100
6,O006,C005,Esther,Corporate,15000
7,O007,C006,Faith,SME,7600
8,O008,C002,Brian,Corporate,8300
9,O009,C007,George,Retail,2500


In [28]:
print(len(orders))
print(len(joined)) 

15
16


Q8. Are the row counts equal? If not, explain exactly what caused the difference. 
-- No the row counts are not equal. The difference was caused by having the same customer being recorded twice in the customer table 

Q9. Locate the unmatched customer record in the joined output. What values appear for CustomerName and 
Segment? 
-- CustomerName: David 
-- Segment: SME

Reconciling the Totals 

In [ ]:
# Before the join:
orders["OrderAmount"].sum() 

np.int64(113500)

In [30]:
# After the join: 
joined["OrderAmount"].sum() 

np.int64(122600)

Q10. Are the two sales totals equal? 
-- No they are not 

Q11. If the total changed, which customer caused the duplication? 
-- Customer with CustomerID : C004

Q12. Why is total reconciliation an important validation step after a JOIN? 
-- To ensure duplicate values do not inflate theh value of the totals 

Correct the Duplicate-Key Problem

In [ ]:
customers_clean = ( 
    customers 
    .drop_duplicates(subset="CustomerID") 
) 


customers_clean.to_sql( 
    "customers_clean", 
    conn, 
    if_exists="replace", 
    index=False 
) 



query = """ 
SELECT 
    o.OrderID, 
    o.CustomerID, 
    c.CustomerName, 
    c.Segment, 
    o.OrderAmount 
FROM orders o 
LEFT JOIN customers_clean c 
    ON o.CustomerID = c.CustomerID; 
""" 
 
clean_joined = pd.read_sql_query(query, conn) 
clean_joined 


,OrderID,CustomerID,CustomerName,Segment,OrderAmount
0,O001,C001,Amina,Retail,4500
1,O002,C002,Brian,Corporate,12500
2,O003,C001,Amina,Retail,3200
3,O004,C003,Carol,Retail,6800
4,O005,C004,David,SME,9100
5,O006,C005,Esther,Corporate,15000
6,O007,C006,Faith,SME,7600
7,O008,C002,Brian,Corporate,8300
8,O009,C007,George,Retail,2500
9,O010,C008,Halima,Corporate,11000


In [34]:
# Verify Totals 
print(len(orders)), print(len(clean_joined)) 
orders["OrderAmount"].sum(), clean_joined["OrderAmount"].sum() 

15
15


(np.int64(113500), np.int64(113500))

Q13. Has the corrected JOIN preserved both the row count and the total sales value? Explain. 
-- Yes! Because the corrected join does not contain any duplicate values in the left column that is being joined, the row counts before and after the join are the same. As well as the totals before and after as a result. 

Customer Segment KPIs

In [35]:
query = """ 
    SELECT 
        COALESCE(c.Segment, 'Unmatched / NULL') AS Segment,
        COUNT(o.OrderID) AS Orders,
        SUM(o.OrderAmount) AS "Total Sales",
        AVG(o.OrderAmount) AS "Average Order Value"
    FROM orders o
    LEFT JOIN customers_clean c 
        ON o.CustomerID = c.CustomerID
    GROUP BY COALESCE(c.Segment, 'Unmatched / NULL')
""" 
 
pd.read_sql_query(query, conn)

,Segment,Orders,Total Sales,Average Order Value
0,Corporate,4,46800,11700.000000
1,Retail,6,29600,4933.333333
2,SME,4,27200,6800.000000
3,Unmatched / NULL,1,9900,9900.000000


Q14. Write three sentences explaining what management can learn from the segment KPIs. 
-- 1. The most profitable customer segment is Corporate, with the highest Total Sales value of ksh 46,800/= and highest average order value of ksh 11,700/=
-- 2. The Retail customer Segment has the highest amount of transactions but a lower average order value than Corporate of ksh 4,933/= 
-- 3. There is one unmatched transaction, leaving  ksh 9900 without a segment to be assigned to.

Merge the JSON Region Lookup 

In [42]:
# Load Region Data
regions_df = pd.read_json("regions.json") 
regions_df 


,RegionCode,RegionName
0,NRB,Nairobi
1,MSA,Mombasa
2,KSM,Kisumu
3,NKR,Nakuru


In [40]:
# Merge Customers with Regions
customers_regions = customers_clean.merge( 
    regions_df, 
    on="RegionCode", 
    how="left" 
) 
 
customers_regions.head()

,CustomerID,CustomerName,Segment,RegionCode,RegionName
0,C001,Amina,Retail,NRB,Nairobi
1,C002,Brian,Corporate,MSA,Mombasa
2,C003,Carol,Retail,NRB,Nairobi
3,C004,David,SME,KSM,Kisumu
4,C005,Esther,Corporate,NKR,Nakuru


Q15. Confirm that each customer now has a readable RegionName rather than only a RegionCode. 
-- Yes, that is now the case 

Q16. If a RegionCode did not exist in regions.json, what would a left merge produce for RegionName?
-- It would produce a NaN for those columns with those region codes 

Regional Sales Analysis 

In [44]:
# 1. Save regions_df into the SQLite database
regions_df.to_sql(
    "regions_df", 
    conn, 
    if_exists="replace", 
    index=False
)

query = """ 
    SELECT 
        r.RegionName AS Region,
        COUNT(o.OrderID) AS "Number of Orders",
        SUM(o.OrderAmount) AS "Total Sales"
    FROM orders o
    JOIN customers_clean c ON o.CustomerID = c.CustomerID
    JOIN regions_df r ON c.RegionCode = r.RegionCode
    GROUP BY r.RegionName
""" 
 
pd.read_sql_query(query, conn)

,Region,Number of Orders,Total Sales
0,Kisumu,2,20100
1,Mombasa,3,23300
2,Nairobi,7,39800
3,Nakuru,2,20400


Q17. Which region generated the highest sales? 
-- Nairobi with ksh 39,800

Q18. Should management automatically conclude that this is the “best-performing” region? Explain using at 
least two limitations of the available data. 
-- No they should not 

-- The gross revenue data doesn't account for the cost to serve, therefore this doesn't show what the most profitable region is when you account for all the expenses inqured for the different regions 

-- The higher sales volume seen in the Nairobi region could also be because it has a larger customer base than other regions with a smaller economic hub.

Write a conclusion of no more than five sentences covering all of the following: 
• The biggest data-quality problem you found. 
• How it affected the JOIN. 
• Why totals should be checked after joining tables. 
• Which customer segment generated the most sales. 
• One limitation of the analysis.


-- The biggest data-quality problem I found was the presence of duplicate       records within the customer table
It inflated metrics and caused unexpected row-multiplications during table joins 
Because of this, totals should always be validated after joining tables in order to prevent incorrect reporting. 
The Corporate segment was the one that generated the most sales within the customer base. 
With all that said, one limititation of this analysis is that Gross revenue alone does not account for profitability or the cost to serve. 